In [2]:
from dotenv import load_dotenv
load_dotenv()


from anthropic import Anthropic


client = Anthropic()
model = "claude-haiku-4-5"

In [3]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [5]:

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output


In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }


In [7]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results


In [9]:
import json

with open("08-dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)


In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Python Function to Extract AWS Region from S3 Bucket URL\n\nHere are several solutions, from simple to more robust:\n\n## Solution 1: Simple Regex (Recommended)\n\n```python\nimport re\n\ndef extract_region_from_s3_url(url: str) -> str:\n    \"\"\"\n    Extract AWS region from an S3 bucket URL.\n    \n    Examples:\n        's3://my-bucket.s3.us-west-2.amazonaws.com/key' -> 'us-west-2'\n        's3://my-bucket.s3.amazonaws.com/key' -> 'us-east-1' (default region)\n        's3://my-bucket.s3-us-west-2.amazonaws.com/key' -> 'us-west-2'\n    \n    Args:\n        url: S3 bucket URL string\n        \n    Returns:\n        Region code (e.g., 'us-west-2') or 'us-east-1' as default\n    \"\"\"\n    # Pattern for s3.region.amazonaws.com or s3-region.amazonaws.com\n    match = re.search(r's3[.-]([a-z0-9-]+)\\.amazonaws\\.com', url)\n    \n    if match:\n        region = match.group(1)\n        # If region is just 's3', it's the default us-east-1\n        return 'us-east-1'